# Lección 3.5 — Variantes: batch, estocástico y mini-batch

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 90–110 min (lección ≈ 60 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente) y Lección 3.4 (tasa de aprendizaje, épocas y tamaño de lote).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir el descenso de gradiente **batch** (por lotes completos), **estocástico** (SGD) y **mini-batch**.
- Explicar por qué un lote pequeño da una dirección **ruidosa**, y ver ese ruido en una gráfica.
- Comparar las tres variantes por **pasos**, por **épocas** y por **tiempo**, y entender por qué casi todo el mundo usa mini-batch.
- Reconocer dos trampas clásicas: **no barajar** los datos y una tasa **fija** con SGD.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Los 1,000 días de la cadena de heladerías de la Lección 3.4 (temperatura escalada) y una versión ampliada de nuestra máquina de entrenar, que ahora también guarda el **camino** de las perillas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

plt.rcParams["figure.figsize"] = (8, 4.5)
np.seterr(all="ignore")

rng = np.random.default_rng(42)
N = 1000
temperatura = rng.uniform(14, 36, N)
ventas = 8 * temperatura - 60 + rng.normal(0, 15, N)
X = ((temperatura - temperatura.mean()) / temperatura.std()).reshape(-1, 1)

def mse(real, pred):
    return np.mean((pred - real) ** 2)

def entrenar(X, y, tasa, epocas, tam_lote, barajar=True, decaimiento=0.0, semilla=0, max_pasos=None):
    rng_ent = np.random.default_rng(semilla)
    n, p = X.shape
    w, b = np.zeros(p), 0.0
    r = {"perdida_iter": [mse(y, X @ w + b)], "camino": [(w[0], b)], "perdida_epoca": [mse(y, X @ w + b)]}
    paso = 0
    for epoca in range(epocas):
        orden = rng_ent.permutation(n) if barajar else np.arange(n)      # ¿barajamos en cada época?
        for inicio in range(0, n, tam_lote):
            if max_pasos is not None and paso >= max_pasos:
                break
            lote = orden[inicio:inicio + tam_lote]
            t = tasa / (1 + decaimiento * paso)                          # tasa que baja con el tiempo (Sección 5)
            error = X[lote] @ w + b - y[lote]
            w = w - t * 2 * X[lote].T @ error / len(lote)
            b = b - t * 2 * error.mean()
            paso += 1
            r["perdida_iter"].append(mse(y, X @ w + b))
            r["camino"].append((w[0], b))
        r["perdida_epoca"].append(mse(y, X @ w + b))
    r["w"], r["b"] = w, b
    r["perdida_iter"], r["camino"] = np.array(r["perdida_iter"]), np.array(r["camino"])
    return r

print("✅ Todo listo")

---
## 1. 📖 La intuición: ¿a cuánta gente le preguntas antes de dar un paso?

Imagina que diriges una empresa y antes de cada decisión haces una **encuesta** para saber hacia dónde ir.

| Variante | Antes de cada paso le preguntas a… | Ventaja | Desventaja |
|---|---|---|---|
| **Batch** (*batch gradient descent*) | **Todo el país** | La dirección es exacta | Cada encuesta tarda muchísimo: das muy pocos pasos |
| **Estocástico** (*SGD*, *stochastic gradient descent*) | **Una persona** al azar | Decides al instante: das muchísimos pasos | Una sola opinión puede ser rarísima: caminas como alguien que se marea |
| **Mini-batch** | Un **grupito** al azar (32, 64, 128 personas…) | Dirección bastante fiable y pasos rápidos | Hay que elegir el tamaño del grupo |

"Estocástico" significa "con azar": la dirección depende de **qué ejemplo te tocó**.

> 📝 **Ojo con los nombres:** en la práctica, cuando una librería dice "SGD" casi siempre se refiere a **mini-batch**. Y "batch" a veces se usa para cualquier lote (como en `batch_size`). Aquí usaremos los tres nombres en su sentido estricto.

### ✋ Ejercicio 1 — ¿Qué variante es?
Con 1,000 días de datos, ¿qué variante se usa en cada caso?
1. `tam_lote = 1000`
2. `tam_lote = 1`
3. `tam_lote = 64`
4. Un modelo que se actualiza **cada vez que llega un nuevo cliente** a la tienda, sin esperar a juntar más.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Batch** (todos los datos en cada paso).  2. **Estocástico** (SGD).  3. **Mini-batch**.  4. **Estocástico**: se le llama también **aprendizaje en línea** (*online learning*). Es útil cuando los datos llegan de uno en uno y no puedes guardarlos todos.
</details>

---
## 2. 📖 El concepto: un lote pequeño da una brújula temblorosa

El gradiente "verdadero" es el que se calcula con **todos** los datos. Con un lote pequeño obtenemos una **estimación**: a veces apunta bien, a veces se desvía un poco, según qué ejemplos te tocaron. Es como una encuesta con pocas personas: el promedio de 1,000 opiniones es más fiable que el de 10.

Hagamos el experimento: nos paramos en un punto del paisaje (`w = 30`, `b = 100`) y calculamos el gradiente con **300 lotes distintos** elegidos al azar, de cada tamaño.

In [ ]:
w0, b0 = 30.0, 100.0
rng_lotes = np.random.default_rng(1)

def gradiente(idx):
    error = X[idx, 0] * w0 + b0 - ventas[idx]
    return 2 * np.mean(error * X[idx, 0]), 2 * np.mean(error)

g_real = gradiente(np.arange(N))
print(f"Gradiente con TODOS los datos: pendiente_w = {g_real[0]:.1f}, pendiente_b = {g_real[1]:.1f}\n")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, tam in zip(axes, [1, 10, 100]):                     # ✏️ prueba otros tamaños
    g = np.array([gradiente(rng_lotes.choice(N, tam, replace=False)) for _ in range(300)])
    ax.scatter(g[:, 0], g[:, 1], s=8, alpha=.5, label=f"300 lotes de {tam}")
    ax.plot(*g_real, "r*", ms=18, label="Gradiente real")
    ax.set_title(f"Lote de {tam}"); ax.set_xlabel("pendiente_w"); ax.legend(loc="lower right"); ax.grid(alpha=.3)
    print(f"Lote de {tam:>3}: dispersión de pendiente_w = {g[:, 0].std():6.1f} · de pendiente_b = {g[:, 1].std():5.1f}")
axes[0].set_ylabel("pendiente_b"); plt.show()

### 🔮 Ejercicio 2 — Mide el temblor
1. ¿Las nubes de puntos están **centradas** en la estrella roja (el gradiente real)?
2. Pasar de lotes de 1 a lotes de 100 cuesta **100 veces** más cálculo por paso. ¿Cuántas veces se reduce la dispersión? ¿Es buen negocio?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sí**: en promedio, los lotes apuntan hacia el mismo lugar que el gradiente real. No están "sesgados": solo **tiemblan** alrededor de la dirección correcta.
2. La dispersión se reduce unas **10 veces**, no 100 (en las tres gráficas cae de forma aproximada como 1 / √tamaño del lote: √100 = 10). Cada vez que multiplicas el lote por 100 pagas 100 veces más cálculo para reducir el ruido solo 10 veces. **Rendimientos decrecientes**: por eso nadie usa lotes enormes, y los lotes medianos (32–256) suelen ser el punto dulce.
</details>

---
## 3. 📖 Cómo se mueve cada variante

Ahora miremos el **camino** del excursionista sobre el mapa de la pérdida. Para que la comparación sea justa, los tres dan **exactamente 60 pasos** con la **misma tasa** (0.05). Lo único que cambia es cuántos días mira en cada paso.

### 🔮 Ejercicio 3
Antes de ejecutar, dibuja en papel cómo crees que será el camino de cada uno, desde la esquina (0, 0) hasta el fondo del valle.

✍️ *Mi dibujo / descripción:*

In [ ]:
W, B = np.meshgrid(np.linspace(-5, 65, 120), np.linspace(-10, 170, 120))
Z = ((W[..., None] * X[:, 0] + B[..., None] - ventas) ** 2).mean(axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for ax, (nombre, lote) in zip(axes, [("Batch (lote = 1000)", 1000), ("Mini-batch (lote = 32)", 32), ("Estocástico (lote = 1)", 1)]):
    r = entrenar(X, ventas, tasa=0.05, epocas=100, tam_lote=lote, max_pasos=60)
    c = r["camino"]
    ax.contourf(W, B, np.log10(Z), levels=25, cmap="viridis")
    ax.plot(c[:, 0], c[:, 1], ".-", color="tab:red", ms=3, lw=1)
    ax.plot(*c[-1], "r*", ms=14)
    ax.set_title(f"{nombre}\n60 pasos → pérdida {r['perdida_iter'][-1]:.0f}"); ax.set_xlabel("w")
axes[0].set_ylabel("b"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Batch:** un camino **liso**, casi una línea recta hacia el fondo. Cada paso usa la dirección exacta.
- **Mini-batch (32):** casi igual de directo, con **pequeños temblores**.
- **Estocástico (1):** un camino **zigzagueante**, como alguien mareado… ¡pero que también llega a la zona del fondo! Y al llegar no se queda quieto: **sigue bailando** alrededor del mínimo.

Con el mismo número de pasos, batch es el más preciso. Pero el número de pasos no es lo único que importa: cada paso batch **cuesta** 1,000 veces más que uno estocástico. Eso lo vemos ahora.
</details>

---
## 4. 📖 ¿Quién gana? Depende de qué cuentes

Hay tres formas honestas de comparar:
- **Por paso:** ¿quién baja más con cada actualización? (gana batch: su dirección es exacta)
- **Por época:** ¿quién baja más tras mirar los datos **una vez**? (cada época le da a SGD 1,000 pasos y a batch solo 1)
- **Por tiempo:** ¿quién baja más por segundo de cómputo? (lo que de verdad le importa a tu jefe)

### 🔮 Ejercicio 4
Entrenamos **3 épocas**, cada variante con una tasa razonable para ella. ¿Quién crees que ganará en **pérdida tras 3 épocas**? ¿Y quién tardará más **por época**?

✍️ *Tu respuesta:*

In [ ]:
configs = [("Batch (1000)", 1000, 0.3), ("Mini-batch (32)", 32, 0.05), ("Estocástico (1)", 1, 0.005)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
for nombre, lote, tasa in configs:
    inicio = time.perf_counter()
    r = entrenar(X, ventas, tasa=tasa, epocas=3, tam_lote=lote)
    seg = time.perf_counter() - inicio
    pi = r["perdida_iter"]
    a1.plot(np.linspace(0, 3, len(pi)), pi, label=nombre)
    a2.plot(np.arange(len(pi)), pi, label=nombre)
    print(f"{nombre:<16} tasa={tasa:<6} {len(pi) - 1:>5} pasos · pérdida tras 3 épocas = {r['perdida_epoca'][-1]:>8.1f} · {seg:.2f} s")
a1.set_yscale("log"); a1.set_xlabel("Épocas"); a1.set_ylabel("Pérdida (log)"); a1.set_title("Por época"); a1.legend(); a1.grid(alpha=.3)
a2.set_yscale("log"); a2.set_xscale("log"); a2.set_xlabel("Pasos (log)"); a2.set_title("Por paso"); a2.legend(); a2.grid(alpha=.3)
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Por época:** mini-batch y SGD ya están en el fondo (≈ 232) tras 3 épocas; batch, con solo 3 pasos, todavía no llegó (≈ 320) aunque usa la tasa más alta de las tres.
- **Por paso** (panel derecho): batch es el que más baja con cada paso, pero solo dio 3.
- **Tiempo:** en este notebook el estocástico es el **más lento** por época: Python tiene que dar 1,000 pasitos uno por uno, cada uno con su sobrecarga. Batch y mini-batch aprovechan que numpy procesa muchos números **de golpe** (vectorización). En una GPU el efecto es aún mayor: un lote de 64 tarda casi lo mismo que un lote de 1.

📌 Conclusión: **mini-batch combina lo mejor de los dos mundos**: muchos pasos por época (como SGD) y cada paso aprovecha el cálculo en paralelo (como batch), con poco ruido. Por eso es el estándar en deep learning.
</details>

### ✏️ Ejercicio 5 — ¿Cuánto cuesta una época?
Mide el tiempo de **una época** para varios tamaños de lote. Fíjate en cómo cae el tiempo al subir el lote… hasta que deja de caer.

In [ ]:
LOTES = [1, 4, 16, 64, 256, 1000]      # ✏️
for lote in LOTES:
    inicio = time.perf_counter()
    for _ in range(3):                   # repetimos 3 veces para medir mejor
        entrenar(X, ventas, tasa=0.01, epocas=1, tam_lote=lote)
    print(f"lote = {lote:>4} → {(time.perf_counter() - inicio) / 3 * 1000:7.1f} ms por época")

<details><summary><b>👀 Ver respuesta</b></summary>

El tiempo por época cae muchísimo al pasar de lote 1 a lote 16–64 (decenas de veces), y a partir de ~256 casi no mejora. Los números exactos dependen de tu computadora, pero la forma es siempre esta. Con lotes grandes ya ahorraste casi todo el tiempo que se podía ahorrar, pero das **muy pocos pasos por época**. El punto dulce suele estar en los lotes **medianos**.
</details>

---
## 5. 📖 Qué pasa si… usas SGD con la tasa fija

Viste que el estocástico **nunca se queda quieto**: cada paso sigue la opinión de un solo ejemplo, así que, aunque esté en el fondo, el siguiente ejemplo lo empuja hacia algún lado.

La solución clásica es **ir bajando la tasa** con el tiempo (*learning rate decay*): pasos grandes al principio para avanzar rápido y pasos cada vez más pequeños para asentarse. Nuestra función lo hace así: `tasa_del_paso = tasa / (1 + decaimiento × paso)`.

### 🔮 Ejercicio 6
Comparamos SGD con tasa 0.05 **fija** contra la misma tasa **con decaimiento**. ¿Cuál termina con la curva más tranquila? ¿Cuál llega más abajo?

✍️ *Tu respuesta:*

In [ ]:
DECAIMIENTO = 0.005      # ✏️ prueba 0.0005 y 0.05

fija = entrenar(X, ventas, tasa=0.05, epocas=5, tam_lote=1)
baja = entrenar(X, ventas, tasa=0.05, epocas=5, tam_lote=1, decaimiento=DECAIMIENTO)
plt.plot(fija["perdida_iter"], lw=.7, label="Tasa fija 0.05")
plt.plot(baja["perdida_iter"], lw=.7, label=f"Tasa con decaimiento {DECAIMIENTO}")
plt.yscale("log"); plt.ylim(220, 400); plt.xlabel("Paso"); plt.ylabel("Pérdida (log)"); plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"Tasa fija      → pérdida final = {fija['perdida_iter'][-1]:.1f} · promedio de los últimos 1000 pasos = {fija['perdida_iter'][-1000:].mean():.1f}")
print(f"Con decaimiento → pérdida final = {baja['perdida_iter'][-1]:.1f} · promedio de los últimos 1000 pasos = {baja['perdida_iter'][-1000:].mean():.1f}")
print(f"(Tasa al final con decaimiento: {0.05 / (1 + DECAIMIENTO * 5000):.4f})")

<details><summary><b>👀 Ver respuesta</b></summary>

Con la tasa **fija**, la pérdida sigue **saltando** hasta el final (promedio de los últimos 1,000 pasos ≈ 255, contra ≈ 231 del fondo). **Con decaimiento**, los saltos se apagan poco a poco y se asienta casi en el fondo (≈ 231). Al final su tasa es ~26 veces más pequeña.

- Con decaimiento demasiado **fuerte** (0.05) la tasa se encoge antes de llegar y se queda corta.
- Con decaimiento demasiado **débil** (0.0005) casi no se nota la diferencia.

Las librerías hacen esto automáticamente con los *schedulers* (Lección 9.6). En `SGDRegressor` es el valor por defecto `learning_rate="invscaling"`.
</details>

---
## 6. 📖 La trampa de no barajar

Fíjate en que `entrenar` **baraja** los datos al comienzo de cada época. ¿Importa? Imagina que el archivo de la heladería viene **ordenado por temperatura**: primero todos los días fríos y al final todos los calurosos. Si no barajas, cada mini-lote es un grupo de días "parecidos", y el excursionista recibe opiniones muy sesgadas: primero solo días fríos ("¡baja las ventas!") y luego solo calurosos ("¡súbelas!").

### 🔮 Ejercicio 7
Entrenamos mini-batch (lote 10, tasa 0.05, 5 épocas) con los datos **ordenados por temperatura**, una vez **barajando** y otra **sin barajar**. ¿Cómo crees que se verá la curva sin barajar?

✍️ *Tu respuesta:*

In [ ]:
orden_temp = np.argsort(X[:, 0])
X_ord, y_ord = X[orden_temp], ventas[orden_temp]          # el archivo viene ordenado por temperatura

con = entrenar(X_ord, y_ord, tasa=0.05, epocas=5, tam_lote=10, barajar=True)
sin = entrenar(X_ord, y_ord, tasa=0.05, epocas=5, tam_lote=10, barajar=False)
plt.plot(con["perdida_iter"], label="Barajando en cada época")
plt.plot(sin["perdida_iter"], label="Sin barajar (datos ordenados)")
plt.yscale("log"); plt.xlabel("Paso"); plt.ylabel("Pérdida (log)"); plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"Pérdida al final de cada época, barajando   : {np.round(con['perdida_epoca'][1:], 0)}")
print(f"Pérdida al final de cada época, sin barajar : {np.round(sin['perdida_epoca'][1:], 0)}")
print(f"Sin barajar → w = {sin['w'][0]:.1f}, b = {sin['b']:.1f}   ·   barajando → w = {con['w'][0]:.1f}, b = {con['b']:.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sin barajar, la curva forma **escalones** que se repiten en cada época (una caída brusca al empezar cada época, y luego tramos lentos): el modelo persigue a los días fríos, luego a los templados, luego a los calurosos… En lugar de bajar en línea recta hacia el fondo, va dando tumbos según el grupo de días que le toca. Resultado: **aprende muchísimo más lento**. Al final de la primera época, barajando ya está en ≈ 233; sin barajar sigue en ≈ **10,200**, y necesita 4–5 épocas para alcanzar lo que barajando logró en una. Y aun así termina un poco **sesgado hacia los últimos días que vio** (los más calurosos): su `b` queda algo más alto.

**Moraleja:** baraja siempre los datos entre épocas (las librerías lo hacen por defecto: `shuffle=True`). La única excepción son las **series de tiempo** cuando el orden *es* la información, y ahí la validación también cambia (Lección 4.6).
</details>

---
## 7. 📖 Tabla comparativa y perillas

| | **Batch** | **Mini-batch** | **Estocástico (SGD)** |
|---|---|---|---|
| Ejemplos por paso | Todos (N) | Un grupo (16–512) | 1 |
| Pasos por época | 1 | N / lote | N |
| Ruido de la dirección | Ninguno | Poco | Mucho |
| Camino | Liso | Ligeramente tembloroso | Zigzag "mareado" |
| Memoria necesaria | Todos los datos a la vez | Un lote | Un ejemplo |
| Aprovecha el cálculo en paralelo (GPU) | Sí | **Sí** | No |
| ¿Se asienta en el fondo? | Sí | Casi (mejor con decaimiento) | No, salvo con decaimiento |
| Uso típico | Datos pequeños, modelos clásicos | **Deep learning, casi todo** | Datos que llegan uno a uno (*online*) |

### Perillas de esta lección

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Tamaño de lote** (`batch_size`) | Ejemplos por paso | Menos ruido, menos pasos por época, más memoria; puede necesitar subir la tasa | Más ruido, más pasos; puede necesitar bajar la tasa | 32–256 |
| **Barajar** (`shuffle`) | Si se mezcla el orden en cada época | — | Si lo desactivas con datos ordenados: escalones, lentitud y sesgo | `True` (salvo series de tiempo) |
| **Decaimiento de la tasa** (*schedule*) | Cuánto se encoge la tasa con el tiempo | Se asienta antes, pero puede frenar antes de llegar | Avanza más, pero sigue "bailando" | El *scheduler* por defecto de tu librería (Lección 9.6) |

> 💡 **Un efecto bueno del ruido:** el temblor de SGD y mini-batch puede ayudar a **escapar** de mesetas y de valles poco profundos. Lo verás en la próxima lección.

### ✋ Ejercicio 8 — Elige la variante
1. Tienes 500 filas y una regresión lineal en tu laptop.
2. Entrenas una red neuronal con 1 millón de imágenes en una GPU que solo tiene memoria para 256 imágenes a la vez.
3. Un sistema antifraude recibe transacciones una por una y debe adaptarse al instante.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Batch** está bien: con 500 filas cada paso es baratísimo y la dirección es exacta. (De hecho, para una regresión lineal pequeña incluso hay una fórmula directa: `LinearRegression`.)
2. **Mini-batch** con lote ≤ 256: no cabe todo en memoria y quieres aprovechar la GPU.
3. **Estocástico / aprendizaje en línea**: se actualiza con cada transacción que llega.
</details>

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida **nunca se estabiliza**, sigue saltando al final | Lote muy pequeño con tasa fija | Bajar la tasa con decaimiento, o subir el lote |
| Curva con un patrón (escalones, sierra) que se repite cada época | Datos ordenados y **sin barajar** | `shuffle=True` |
| Cambiaste el lote y el entrenamiento empeoró o explotó | La tasa buena depende del lote | Repetir el barrido de tasa (Lección 3.4) |
| "Out of memory" en la GPU | Lote demasiado grande | Bajar el lote (y quizá la tasa) |
| Comparas variantes "por época" y concluyes que batch es malísimo | Batch da 1 paso por época: la comparación no es justa | Compara también por pasos y por tiempo |
| Una época con lote = 1 tarda una eternidad | Sin vectorización, cada ejemplo paga su propio costo fijo | Usa mini-batch |

## 9. 🧠 Resumen en 5 puntos
1. **Batch** usa todos los datos por paso (exacto, pocos pasos); **estocástico** usa uno (muchos pasos, mucho ruido); **mini-batch** usa un grupo (el equilibrio).
2. Un lote pequeño da un gradiente **ruidoso pero sin sesgo**: tiembla alrededor de la dirección correcta. El ruido baja como 1/√lote.
3. Mini-batch es el estándar porque da **muchos pasos por época** y cada paso **aprovecha el cálculo en paralelo**.
4. Con SGD o lotes chicos, una **tasa que decae** ayuda a asentarse en el fondo.
5. **Baraja** siempre los datos entre épocas (salvo series de tiempo).

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** Con 10,000 ejemplos y SGD (lote = 1), ¿cuántos pasos hay en una época?
- a) 1 · b) 100 · c) 10,000 · d) Depende de la tasa

**2.** Multiplicas el tamaño de lote por 4. ¿Qué le pasa aproximadamente al ruido del gradiente?
- a) Se divide entre 4 · b) Se divide entre 2 · c) No cambia · d) Se multiplica por 4

**3.** ¿Por qué el camino de SGD no se queda quieto en el fondo con tasa fija?
- a) Porque tiene un error · b) Porque cada paso sigue la opinión de un solo ejemplo, que nunca coincide del todo con el promedio · c) Porque el fondo se mueve · d) Porque no baraja

**4.** Ves una curva de pérdida con un patrón de escalones que se repite exactamente una vez por época. ¿Qué revisas primero?
- a) La tasa · b) Si se están barajando los datos · c) El número de épocas · d) La función de pérdida

**5.** En una librería de deep learning, el optimizador se llama "SGD" y le pasas `batch_size=64`. ¿Qué variante usas realmente?
- a) Batch · b) Estocástico puro · c) Mini-batch · d) Ninguna

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Un paso por ejemplo.
2. **b)** El ruido baja como 1/√lote: √4 = 2.
3. **b)**
4. **b)** Un patrón que se repite en cada época apunta al orden de los datos.
5. **c)** El nombre "SGD" en las librerías casi siempre es mini-batch.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Una amiga dice: *"El ruido de mini-batch es un defecto: si tuviera una GPU infinita, siempre usaría batch completo"*. ¿Estás de acuerdo? Piensa en cuántos pasos se dan por época y en lo que adelantamos sobre los valles poco profundos.

**B.** Entrenas con lote 32 y todo va bien. Para acelerar, pasas a lote 1,024 (32 veces más grande) **sin cambiar** la tasa ni las épocas. El entrenamiento termina mucho antes, pero el modelo queda peor. ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** No del todo. Con batch completo das **un solo paso por época**: aunque cada paso sea perfecto, avanzas poco. Además, el ruido de los lotes pequeños puede ayudar a **escapar de mesetas y mínimos poco profundos** (Lección 3.6), y en la práctica se ha observado que los modelos entrenados con lotes moderados a menudo **generalizan** mejor. El ruido, en dosis adecuadas, es un aliado.

**B.** Con lote 1,024 se dan **32 veces menos pasos** en las mismas épocas. Con la misma tasa, el excursionista camina 32 veces menos y se queda a media ladera. Para compensar suele hacer falta **subir la tasa** (y revisar con un barrido) o **entrenar más épocas**.
</details>

### 10.3 🧩 Práctica integradora: una tienda en línea con 20,000 pedidos 🛒

Una tienda quiere predecir el **monto de cada pedido** a partir de 5 características (ya escaladas). Tienes que elegir **cómo entrenar**: batch, mini-batch o estocástico.

**Paso 1 ▶️ — Los datos y la meta**

In [ ]:
rng_t = np.random.default_rng(11)
N_T = 20_000
X_t = rng_t.normal(size=(N_T, 5))
monto = X_t @ np.array([40, -25, 15, 0, 8]) + 300 + rng_t.normal(0, 30, N_T)

from sklearn.linear_model import LinearRegression
meta_t = mse(monto, LinearRegression().fit(X_t, monto).predict(X_t))
print(f"{N_T} pedidos · pérdida mínima posible (meta) = {meta_t:.1f}")

def entrenar_rapido(X, y, tasa, epocas, tam_lote, barajar=True, semilla=0):
    # Igual que entrenar(), pero solo mide la pérdida al final (con 20,000 filas, medirla en cada paso sería muy lento)
    rng_ent = np.random.default_rng(semilla)
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epocas):
        orden = rng_ent.permutation(len(y)) if barajar else np.arange(len(y))
        for inicio in range(0, len(y), tam_lote):
            lote = orden[inicio:inicio + tam_lote]
            error = X[lote] @ w + b - y[lote]
            w = w - tasa * 2 * X[lote].T @ error / len(lote)
            b = b - tasa * 2 * error.mean()
    return mse(y, X @ w + b)

**Paso 2 ✋ — Cuenta los pasos.** Para **una época**, ¿cuántos pasos da batch, mini-batch con lote 64 y estocástico?

✍️ *Mis cuentas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Batch: **1** paso.  - Mini-batch (64): 20,000 / 64 = 312.5 → **313** pasos.  - Estocástico: **20,000** pasos.
</details>

**Paso 3 🔮 — Una época cada uno.** Entrenamos **una sola época** con estas tasas: batch 0.1, mini-batch 0.05, estocástico 0.005. Predice el orden en pérdida (de mejor a peor) y en tiempo (de más rápido a más lento).

✍️ *Mi predicción:*

In [ ]:
for nombre, lote, tasa in [("Batch", N_T, 0.1), ("Mini-batch (64)", 64, 0.05), ("Estocástico", 1, 0.005)]:
    inicio = time.perf_counter()
    p = entrenar_rapido(X_t, monto, tasa=tasa, epocas=1, tam_lote=lote)
    print(f"{nombre:<16} → pérdida = {p:>10,.1f} ({p / meta_t - 1:+8.1%} vs. meta) · {time.perf_counter() - inicio:5.2f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Pérdida:** mini-batch queda prácticamente en la meta (+0.3 %); el estocástico también baja mucho pero queda algo por encima (≈ +2 %) porque sigue "bailando"; batch, con **un solo paso**, queda altísimo (≈ 60,000).
- **Tiempo:** batch y mini-batch tardan centésimas de segundo; el estocástico es el más lento con diferencia (sus 20,000 pasitos de Python), aunque los segundos exactos dependen de tu computadora.

Mini-batch gana en las dos cosas a la vez.
</details>

**Paso 4 ✏️ — Dale a batch lo que necesita.** ¿Cuántas épocas necesita batch (tasa 0.1) para quedar a menos de 1 % de la meta? Cambia `EPOCAS` hasta lograrlo y compara el tiempo con el de mini-batch del Paso 3.

In [ ]:
EPOCAS = 5          # ✏️
inicio = time.perf_counter()
p = entrenar_rapido(X_t, monto, tasa=0.1, epocas=EPOCAS, tam_lote=N_T)
print(f"Batch, {EPOCAS} épocas → pérdida = {p:,.1f} ({p / meta_t - 1:+.2%} vs. meta) · {time.perf_counter() - inicio:.2f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

Con 5 épocas todavía queda lejísimos (≈ 10,900). Necesita **entre 20 y 25 épocas** (con 20 queda a +1.4 % de la meta; con 25, a +0.15 %). Es decir: para llegar al mismo sitio, batch tiene que **leer los datos más de 20 veces** y mini-batch solo **una**. Con 20,000 filas aún es rápido; con 20 millones, esa diferencia son horas.
</details>

**Paso 5 🔮 — La trampa del archivo ordenado.** El sistema exportó los pedidos **ordenados por monto**. Entrenas mini-batch (64, tasa 0.05) **sin barajar** durante 3 épocas. ¿Qué esperas?

✍️ *Tu respuesta:*

In [ ]:
orden_monto = np.argsort(monto)
for barajar in [True, False]:
    p = entrenar_rapido(X_t[orden_monto], monto[orden_monto], tasa=0.05, epocas=3, tam_lote=64, barajar=barajar)
    print(f"barajar={barajar!s:<5} → pérdida = {p:,.1f} ({p / meta_t - 1:+.1%} vs. meta)")

<details><summary><b>👀 Ver respuesta</b></summary>

Barajando llega a la meta. **Sin barajar**, la pérdida queda **muy por encima** (cientos de por ciento): cada época termina con los pedidos más caros, así que el modelo acaba convencido de que todos los pedidos son caros. Es la misma trampa de la Sección 6, pero aquí es aún peor porque los datos están ordenados **por la respuesta**.
</details>

**Paso 6 ✋ — Tu recomendación.** En 3 frases para el equipo de datos: ¿qué variante y qué tamaño de lote usarías, y qué precaución tomarías?

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"Usaremos mini-batch con lotes de 64: llegó a la mejor pérdida posible leyendo los datos una sola vez y fue tan rápido como el batch completo por época. Batch necesitó más de 20 lecturas para llegar al mismo punto, y el estocástico fue el más lento y quedó temblando por encima del óptimo. Siempre barajaremos los pedidos en cada época, porque el archivo viene ordenado por monto y sin barajar el modelo queda inservible."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** Andrew Ng, *"Mini-batch gradient descent"* y *"Understanding mini-batch gradient descent"* (inglés, con subtítulos).
- 📘 **Lectura:** Sebastian Ruder, *"An overview of gradient descent optimization algorithms"* (blog, 2016).
- 📘 **Documentación:** scikit-learn, *Stochastic Gradient Descent* (guía de usuario): `SGDRegressor`, `SGDClassifier` y sus opciones `shuffle` y `learning_rate`.

---
⬅️ **Anterior:** [3.4 Hiperparámetros del entrenamiento: tasa de aprendizaje, épocas, tamaño de lote e iteraciones](3.4-hiperparametros-del-entrenamiento.ipynb) · ➡️ **Siguiente:** [3.6 Mínimos locales, mesetas y por qué no siempre importan](3.6-minimos-locales-y-mesetas.ipynb)